# DermSense AI: Skin Lesion Classification Pipeline

This notebook demonstrates how to run the modular training and evaluation pipeline for skin disease classification (HAM10000 dataset). It includes:
1. **Traditional ML baseline**: SVM with RBF kernel trained on color histograms & HOG features.
2. **Deep Learning model**: EfficientNet-B0 fine-tuned on dermoscopic images.
3. **Model explainability**: Grad-CAM overlays to visualize regions of interest.

In [ ]:
%matplotlib inline
import os
import pandas as pd
import matplotlib.pyplot as plt
import torch

from src.config import DATASET_DIR, BEST_MODEL_PATH, device, TRAIN_CFG
from src.dataset import ensure_ham10000_present, organize_by_class, get_dataset_splits, get_dataloaders
from src.svm_model import train_and_eval_svm
from src.deep_learning import setup_efficientnet_model, train_skin_classifier, eval_cnn_on_test
from src.utils import plot_training_history, plot_and_save_confusion_matrix, plot_and_save_roc_curves

## 1. Load and Organize the Dataset

This script programmatically downloads the HAM10000 dataset from public repositories (Harvard Dataverse fallback to Kaggle if credentials are set) and organizes them into class folders.

In [ ]:
ensure_ham10000_present()

meta_path = None
for candidate in ['HAM10000_metadata.csv', 'HAM10000_metadata.tab', 'HAM10000_metadata', 'hmnist_28_28_RGB.csv']:
    p = os.path.join(DATASET_DIR, candidate)
    if os.path.exists(p):
        meta_path = p
        break

df = pd.read_csv(meta_path, sep='\t' if meta_path.endswith('.tab') else ',')
organize_by_class(df)

## 2. Train Traditional ML Baseline (SVM + HOG)

In [ ]:
full_dataset, train_idx, val_idx, test_idx, class_names = get_dataset_splits()

svm_clf, svm_preds, svm_probs, svm_accuracy, y_test = train_and_eval_svm(
    full_dataset, train_idx, test_idx, class_names
)

## 3. Train Convolutional Neural Network (EfficientNet-B0)

Train a transfer learning model in two phases (phase 1: frozen backbone, train classifier head; phase 2: unfreeze and fine-tune).

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(full_dataset, train_idx, val_idx, test_idx)
num_classes = len(class_names)

model = setup_efficientnet_model(num_classes)
history, phase1_epochs_run = train_skin_classifier(model, train_loader, val_loader)

### Plot Training History

In [ ]:
plot_training_history(history, phase1_epochs_run)
# View saved figure
from PIL import Image
img = Image.open('outputs/cnn_training_history.png')
plt.figure(figsize=(10, 5))
plt.imshow(img)
plt.axis('off')
plt.show()

### Evaluate CNN on Test Dataset

In [ ]:
cnn_preds, cnn_probs, cnn_accuracy, y_test_cnn = eval_cnn_on_test(model, test_loader, class_names)

## 4. Model Explainability with Grad-CAM

In [ ]:
from src.explainability import GradCAM
from src.utils import denormalize
import random
from src.dataset import SubsetWithTransform, get_val_test_transforms

test_dataset = SubsetWithTransform(full_dataset, test_idx, get_val_test_transforms())
target_layer = model.features[-1]
grad_cam = GradCAM(model, target_layer)

# Generate Grad-CAM for a random test image
idx = random.randint(0, len(test_dataset) - 1)
img_tensor, true_label = test_dataset[idx]

cam, pred_class, probs = grad_cam.generate(img_tensor.unsqueeze(0).to(device))
orig_img = denormalize(img_tensor).transpose(1, 2, 0)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(orig_img)
axes[0].axis('off')
axes[0].set_title(f"True: {class_names[true_label]}\nPred: {class_names[pred_class]} ({probs[pred_class]*100:.1f}%)")

axes[1].imshow(orig_img)
axes[1].imshow(cam, alpha=0.5, cmap='jet')
axes[1].axis('off')
axes[1].set_title("Grad-CAM Saliency")
plt.show()